# Simulación · Unidad Modular II
## Generación y modelamiento: generadores, propiedades y pruebas estadísticas

Cuaderno práctico que acompaña la presentación. Cada sección reproduce los ejemplos de las diapositivas con **NumPy**, **pandas**, **Matplotlib**, **seaborn** y **SciPy**.

| Sección | Tema |
|---|---|
| 1 | Generadores no congruenciales: cuadrados medios, productos medios y multiplicador constante |
| 2 | Generadores congruenciales: mixto, multiplicativo, aditivo, cuadrático y Blum, Blum y Shub |
| 3 | Propiedades: media, varianza e independencia |
| 4 | Pruebas de medias y de varianza |
| 5 | Pruebas de uniformidad: chi-cuadrada y Kolmogorov-Smirnov |
| 6 | Pruebas de independencia: corridas y póker |
| 7 | Batería completa y comparación de generadores |
| 8 | ¿Qué significa α? Experimento con 300 réplicas |
| 9 | Ejercicios y comprobación |

Ejecuta las celdas en orden (`Shift + Enter`) o usa *Entorno de ejecución → Ejecutar todas*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy import stats

MORADO, VERDE, AMBAR, CORAL, AZUL, TINTA = "#6C4AB6", "#00A896", "#F2A900", "#E4572E", "#3A86C8", "#1A1433"
sns.set_theme(style="whitegrid", palette=[MORADO, VERDE, CORAL, AZUL, AMBAR])
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.precision", 4)

---
## 1 y 2. Generadores

Todas las funciones devuelven una lista de filas (un diccionario por paso) que se convierte en tabla con `pd.DataFrame`. Para los congruenciales se usa $r_i = X_i / (m-1)$.

In [ ]:
def digitos_centrales(y, D):
    s = str(y).zfill(2 * D)
    if (len(s) - D) % 2:
        s = "0" + s
    ini = (len(s) - D) // 2
    return s, s[ini:ini + D]

def cuadrados_medios(x0, n, D=4):
    filas, x = [], x0
    for i in range(n):
        y, c = digitos_centrales(x * x, D)
        filas.append(dict(i=i, X=str(x).zfill(D), Y=y, Xn=c, r=float("0." + c)))
        x = int(c)
    return filas

def productos_medios(x0, x1, n, D=4):
    filas, a, b = [], x0, x1
    for i in range(n):
        y, c = digitos_centrales(a * b, D)
        filas.append(dict(i=i, X=str(a).zfill(D), X1=str(b).zfill(D), Y=y, Xn=c, r=float("0." + c)))
        a, b = b, int(c)
    return filas

def multiplicador_constante(a, x0, n, D=4):
    filas, x = [], x0
    for i in range(n):
        y, c = digitos_centrales(a * x, D)
        filas.append(dict(i=i, X=str(x).zfill(D), Y=y, Xn=c, r=float("0." + c)))
        x = int(c)
    return filas

def congruencial(a, c, m, x0, n):
    filas, x = [], x0
    for i in range(n):
        t = a * x + c; nx = t % m
        filas.append(dict(i=i, X=x, T=t, Xn=nx, r=nx / (m - 1)))
        x = nx
    return filas

def aditivo(semillas, m, n):
    """X_i = (X_{i-1} + X_{i-k}) mod m, con k = número de semillas."""
    xs, filas, k = list(semillas), [], len(semillas)
    for i in range(n):
        a, b = xs[-1], xs[-k]
        nx = (a + b) % m
        filas.append(dict(i=len(xs) + 1, a=a, b=b, T=a + b, Xn=nx, r=nx / (m - 1)))
        xs.append(nx)
    return filas

def cuadratico(a, b, c, m, x0, n):
    filas, x = [], x0
    for i in range(n):
        t = a * x * x + b * x + c; nx = t % m
        filas.append(dict(i=i, X=x, T=t, Xn=nx, r=nx / (m - 1)))
        x = nx
    return filas

def bbs(p, q, x0, n):
    m, filas, x = p * q, [], x0
    for i in range(n):
        nx = (x * x) % m
        filas.append(dict(i=i, X=x, T=x * x, Xn=nx, bit=nx % 2, r=nx / (m - 1)))
        x = nx
    return filas

def periodo(estados):
    vistos = {}
    for i, e in enumerate(estados):
        if e in vistos:
            return vistos[e], i - vistos[e]
        vistos[e] = i
    return None

def minstd(seed, n):
    out, x = np.empty(n), seed
    for i in range(n):
        x = (16807 * x) % 2147483647; out[i] = x / 2147483647
    return out

tabla = lambda filas: pd.DataFrame(filas).set_index('i')

### 1.1 Multiplicador constante

$$Y_i = a \cdot X_i \qquad X_{i+1} = \text{D dígitos centrales de } Y_i \qquad r_{i+1} = 0.X_{i+1}$$

Ejemplo de la presentación: $a = 6965$, $X_0 = 9803$, $D = 4$.

In [ ]:
tabla(multiplicador_constante(6965, 9803, 6))

### 1.2 ¿Qué tan largos son los periodos de los métodos de dígitos centrales?

Se prueban **todas** las semillas de 4 dígitos (1000 a 9999) con cuadrados medios y con multiplicador constante ($a = 6965$). Se cuenta cuántos valores distintos produce cada semilla antes de repetirse.

In [ ]:
def distintos(filas):
    p = periodo([f["Xn"] for f in filas])
    return len(filas) if p is None else p[0] + p[1]   # None: no se repitió en los pasos generados

semillas = range(1000, 10000)
largos = pd.DataFrame({
    "Cuadrados medios": [distintos(cuadrados_medios(s, 400)) for s in semillas],
    "Multiplicador constante": [distintos(multiplicador_constante(6965, s, 400)) for s in semillas],
})
display(largos.describe().round(1))

largo = largos.melt(var_name="método", value_name="valores distintos")
sns.histplot(data=largo, x="valores distintos", hue="método", bins=40, element="step", stat="count")
plt.title("Valores distintos antes de repetirse (9 000 semillas)"); plt.show()

Ninguna semilla de 4 dígitos llega cerca de 10 000 valores distintos: los métodos no congruenciales se agotan rápido.

### 2.1 Congruencial aditivo

$$X_i = (X_{i-1} + X_{i-n}) \bmod m$$

Semillas 65, 89, 98, 03, 69 y $m = 100$.

In [ ]:
pd.DataFrame(aditivo([65, 89, 98, 3, 69], 100, 7)).set_index("i")

### 2.2 Congruencial cuadrático

$$X_{i+1} = (a X_i^2 + b X_i + c) \bmod m$$

Periodo completo $N = m$ si $m = 2^g$, $a$ par, $c$ impar y $(b - a) \bmod 4 = 1$. Ejemplo: $a = 26$, $b = 27$, $c = 27$, $m = 8$, $X_0 = 13$.

In [ ]:
df = tabla(cuadratico(26, 27, 27, 8, 13, 9))
print("Periodo:", periodo(df["Xn"].tolist())[1])
df

**Mapa de periodos.** Con $m = 16$ y $c = 5$ se calcula el periodo para cada combinación de $a$ y $b$. Las celdas con periodo 16 son exactamente las que cumplen las condiciones.

In [ ]:
m, c = 16, 5
mapa = pd.DataFrame(index=pd.Index(range(m), name="a"), columns=pd.Index(range(m), name="b"), dtype=int)
for a in range(m):
    for b in range(m):
        mapa.loc[a, b] = periodo([f["Xn"] for f in cuadratico(a, b, c, m, 1, 3 * m)])[1]

plt.figure(figsize=(10, 7))
sns.heatmap(mapa.astype(int), annot=True, fmt="d", cmap="Purples", cbar_kws={"label": "periodo"})
plt.title("Periodo del generador cuadrático (m = 16, c = 5)"); plt.show()

cumple = [(a, b) for a in range(m) for b in range(m) if a % 2 == 0 and (b - a) % 4 == 1]
print("¿Todas las combinaciones que cumplen tienen periodo 16?", all(mapa.loc[a, b] == 16 for a, b in cumple))

### 2.3 Blum, Blum y Shub

$$X_{i+1} = X_i^2 \bmod m, \qquad m = p\,q,\quad p \equiv q \equiv 3 \pmod 4$$

Se usa el bit menos significativo (paridad) de cada $X$. Ejemplo: $p = 11$, $q = 23$, $X_0 = 3$.

In [ ]:
df = tabla(bbs(11, 23, 3, 30))
print("Periodo:", periodo(df["Xn"].tolist())[1])
print("Bits generados:", "".join(map(str, df["bit"])))
df.head(6)

---
## 3. Propiedades: media, varianza e independencia

$$E(r) = \int_0^1 r\,dr = \tfrac12 \qquad V(r) = \int_0^1 \left(r - \tfrac12\right)^2 dr = \tfrac1{12}$$

### 3.1 Convergencia de la media y la varianza
Se generan 20 secuencias independientes y se observa cómo $\bar r$ y $S^2$ se acercan a sus valores teóricos al crecer $n$.

In [ ]:
ns = np.unique(np.logspace(1, 4, 25).astype(int))
filas = []
for rep in range(20):
    r = np.random.default_rng(rep).random(ns.max())
    for n in ns:
        filas.append({"n": n, "réplica": rep, "media": r[:n].mean(), "varianza": r[:n].var(ddof=1)})
conv = pd.DataFrame(filas)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.lineplot(data=conv, x="n", y="media", units="réplica", estimator=None, alpha=0.35, color=MORADO, ax=ax[0])
ax[0].axhline(0.5, color=CORAL, lw=2, label="1/2"); ax[0].set_xscale("log"); ax[0].set_title("Media de 20 secuencias"); ax[0].legend()
sns.lineplot(data=conv, x="n", y="varianza", units="réplica", estimator=None, alpha=0.35, color=VERDE, ax=ax[1])
ax[1].axhline(1 / 12, color=CORAL, lw=2, label="1/12"); ax[1].set_xscale("log"); ax[1].set_title("Varianza de 20 secuencias"); ax[1].legend()
plt.tight_layout(); plt.show()

### 3.2 La media muestral es aproximadamente normal

Por el teorema del límite central, $\bar r \approx N\!\left(\tfrac12, \tfrac{1}{12n}\right)$. Se calculan 5 000 medias de $n = 40$ números.

In [ ]:
n = 40
medias = np.random.default_rng(1).random((5000, n)).mean(axis=1)
sns.histplot(medias, bins=50, stat="density", color=MORADO)
x = np.linspace(0.3, 0.7, 300)
plt.plot(x, stats.norm.pdf(x, 0.5, np.sqrt(1 / (12 * n))), color=CORAL, lw=2.5, label="N(1/2, 1/(12n))")
z = 1.96 / np.sqrt(12 * n)
plt.axvspan(0.5 - z, 0.5 + z, color=VERDE, alpha=0.12, label="zona de aceptación (95 %)")
plt.title("Distribución de la media de 40 números"); plt.legend(); plt.show()
print(f"Proporción de medias fuera de la zona: {np.mean(np.abs(medias - 0.5) > z):.3f}  (se espera 0.05)")

### 3.3 Independencia: dispersión y autocorrelación

In [ ]:
buena = minstd(12345, 2000)
mala = np.array([f["r"] for f in congruencial(37, 1, 256, 1, 2000)])

def autocorr(r, k):
    return np.corrcoef(r[:-k], r[k:])[0, 1]

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
ax[0].scatter(buena[:-1][:500], buena[1:][:500], s=6, color=VERDE); ax[0].set_title("MINSTD")
ax[1].scatter(mala[:-1][:256], mala[1:][:256], s=6, color=CORAL); ax[1].set_title("a = 37, c = 1, m = 256")
for a in ax[:2]: a.set(xlabel="r(i)", ylabel="r(i+1)", xlim=(0, 1), ylim=(0, 1))
ac = pd.DataFrame({"retardo k": list(range(1, 11)) * 2,
                   "autocorrelación": [autocorr(buena, k) for k in range(1, 11)] + [autocorr(mala, k) for k in range(1, 11)],
                   "generador": ["MINSTD"] * 10 + ["m = 256"] * 10})
sns.barplot(data=ac, x="retardo k", y="autocorrelación", hue="generador", palette=[VERDE, CORAL], ax=ax[2])
ax[2].axhline(0, color=TINTA, lw=0.8); ax[2].set_title("Autocorrelación por retardo")
plt.tight_layout(); plt.show()

---
## 4 a 6. Pruebas estadísticas

Todas las pruebas usan $\alpha = 0.05$ y devuelven un diccionario con el estadístico, los límites o el valor crítico y la decisión (`acepta = True` significa *no se rechaza* $H_0$).

In [ ]:
def prueba_medias(r, alfa=0.05):
    r = np.asarray(r); n = len(r); z = stats.norm.ppf(1 - alfa / 2)
    media = r.mean(); h = z / np.sqrt(12 * n)
    return dict(n=n, media=media, z=z, LI=0.5 - h, LS=0.5 + h, acepta=bool(0.5 - h <= media <= 0.5 + h))

def prueba_varianza(r, alfa=0.05):
    r = np.asarray(r); n = len(r); s2 = r.var(ddof=1)
    chi_inf = stats.chi2.ppf(alfa / 2, n - 1); chi_sup = stats.chi2.ppf(1 - alfa / 2, n - 1)
    LI, LS = chi_inf / (12 * (n - 1)), chi_sup / (12 * (n - 1))
    return dict(n=n, var=s2, chi_inf=chi_inf, chi_sup=chi_sup, LI=LI, LS=LS, acepta=bool(LI <= s2 <= LS))

def chi2_uniformidad(r, m=None, alfa=0.05):
    r = np.asarray(r); n = len(r); m = m or int(round(np.sqrt(n)))
    O = np.histogram(r, bins=m, range=(0, 1))[0]; E = n / m
    contrib = (O - E) ** 2 / E; chi0 = contrib.sum(); crit = stats.chi2.ppf(1 - alfa, m - 1)
    return dict(n=n, m=m, O=O.tolist(), E=E, contrib=contrib.tolist(), chi0=chi0, crit=crit, acepta=bool(chi0 <= crit))

def ks_critico(n, alfa=0.05):
    return stats.kstwo.ppf(1 - alfa, n)

def prueba_ks(r, alfa=0.05):
    r = np.sort(np.asarray(r)); n = len(r); i = np.arange(1, n + 1)
    dmas = i / n - r; dmenos = r - (i - 1) / n
    D = max(dmas.max(), dmenos.max()); crit = ks_critico(n, alfa)
    return dict(n=n, orden=r.tolist(), dmas=dmas.tolist(), dmenos=dmenos.tolist(), Dmas=dmas.max(), Dmenos=dmenos.max(), D=D, crit=crit, acepta=bool(D <= crit))

def corridas_arriba_abajo(r, alfa=0.05):
    r = np.asarray(r); n = len(r)
    S = [1 if r[i] > r[i - 1] else 0 for i in range(1, n)]
    C = 1 + sum(1 for i in range(1, len(S)) if S[i] != S[i - 1])
    mu = (2 * n - 1) / 3; var = (16 * n - 29) / 90; z0 = (C - mu) / np.sqrt(var); z = stats.norm.ppf(1 - alfa / 2)
    return dict(n=n, S=S, C=C, mu=mu, var=var, z0=z0, z=z, acepta=bool(abs(z0) <= z))

def corridas_media(r, alfa=0.05):
    r = np.asarray(r); n = len(r)
    S = [1 if v >= 0.5 else 0 for v in r]
    C = 1 + sum(1 for i in range(1, n) if S[i] != S[i - 1])
    n1 = sum(S); n0 = n - n1
    mu = 2 * n0 * n1 / n + 0.5; var = 2 * n0 * n1 * (2 * n0 * n1 - n) / (n ** 2 * (n - 1))
    z0 = (C - mu) / np.sqrt(var); z = stats.norm.ppf(1 - alfa / 2)
    return dict(n=n, S=S, C=C, n0=n0, n1=n1, mu=mu, var=var, z0=z0, z=z, acepta=bool(abs(z0) <= z))

MANOS = [("TD", "Todos diferentes", 0.3024), ("1P", "Un par", 0.5040), ("2P", "Dos pares", 0.1080),
         ("T", "Tercia", 0.0720), ("TP", "Tercia y par", 0.0090), ("P", "Póker", 0.0045), ("Q", "Quintilla", 0.0001)]
PATRON = {(1, 1, 1, 1, 1): "TD", (2, 1, 1, 1): "1P", (2, 2, 1): "2P", (3, 1, 1): "T", (3, 2): "TP", (4, 1): "P", (5,): "Q"}

def mano(r):
    d = str(int(r * 100000 + 1e-9)).zfill(5)[:5]
    return PATRON[tuple(sorted(Counter(d).values(), reverse=True))]

def prueba_poker(r, alfa=0.05):
    r = np.asarray(r); n = len(r); cnt = Counter(mano(v) for v in r)
    cats = [[c, nombre, p, cnt.get(c, 0)] for c, nombre, p in MANOS]
    # agrupar las categorías raras (de la última hacia atrás) hasta que E >= 5
    while len(cats) > 2 and n * cats[-1][2] < 5:
        u = cats.pop(); cats[-1] = [cats[-1][0] + "+" + u[0], cats[-1][1] + " o más", cats[-1][2] + u[2], cats[-1][3] + u[3]]
    filas = [dict(cat=c, nombre=nm, p=p, O=o, E=n * p, contrib=(o - n * p) ** 2 / (n * p)) for c, nm, p, o in cats]
    chi0 = sum(f["contrib"] for f in filas); crit = stats.chi2.ppf(1 - alfa, len(filas) - 1)
    return dict(n=n, filas=filas, chi0=chi0, gl=len(filas) - 1, crit=crit, acepta=bool(chi0 <= crit))

### Datos de los ejemplos de la presentación

Con la semilla 10 se reproducen exactamente los números usados en las diapositivas.

In [ ]:
rng = np.random.default_rng(10)
r40 = np.round(rng.random(40), 4)   # medias y varianza
r10 = np.round(rng.random(10), 4)   # Kolmogorov-Smirnov
r20 = np.round(rng.random(20), 4)   # corridas
r100 = rng.random(100)              # chi-cuadrada
r200 = rng.random(200)              # póker
pd.DataFrame(r40.reshape(5, 8))

## 4. Prueba de medias y prueba de varianza

$$LI_{\bar r},\ LS_{\bar r} = \tfrac12 \mp z_{\alpha/2}\frac{1}{\sqrt{12n}} \qquad LI_{S^2} = \frac{\chi^2_{1-\alpha/2,\,n-1}}{12(n-1)},\quad LS_{S^2} = \frac{\chi^2_{\alpha/2,\,n-1}}{12(n-1)}$$

($\chi^2_{\alpha,k}$ deja un área $\alpha$ a la derecha.)

In [ ]:
pm, pv = prueba_medias(r40), prueba_varianza(r40)
resumen = pd.DataFrame([
    {"prueba": "Medias", "estadístico": pm["media"], "LI": pm["LI"], "LS": pm["LS"], "no se rechaza H0": pm["acepta"]},
    {"prueba": "Varianza", "estadístico": pv["var"], "LI": pv["LI"], "LS": pv["LS"], "no se rechaza H0": pv["acepta"]},
]).set_index("prueba")
resumen

In [ ]:
def recta(ax, valor, li, ls, etiqueta, rango):
    ax.axvspan(li, ls, color=VERDE, alpha=0.25, label="zona de aceptación")
    ax.axvline(valor, color=AMBAR if li <= valor <= ls else CORAL, lw=3, label=f"{etiqueta} = {valor:.4f}")
    ax.set_xlim(*rango); ax.set_yticks([]); ax.legend(loc="upper right")

fig, ax = plt.subplots(2, 1, figsize=(10, 3.6))
recta(ax[0], pm["media"], pm["LI"], pm["LS"], "media", (0.35, 0.65)); ax[0].set_title("Prueba de medias")
recta(ax[1], pv["var"], pv["LI"], pv["LS"], "S²", (0.03, 0.15)); ax[1].set_title("Prueba de varianza")
plt.tight_layout(); plt.show()

## 5. Pruebas de uniformidad

### 5.1 Chi-cuadrada con 100 números y $m = \sqrt{100} = 10$ intervalos
$$\chi^2_0 = \sum_{i=1}^{m}\frac{(O_i - E_i)^2}{E_i}, \qquad E_i = n/m$$

In [ ]:
pc = chi2_uniformidad(r100)
tabla_chi = pd.DataFrame({"intervalo": [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)], "O": pc["O"], "E": pc["E"], "(O-E)²/E": pc["contrib"]})
display(tabla_chi.set_index("intervalo").T)
print(f"χ²₀ = {pc['chi0']:.2f}   crítico = {pc['crit']:.2f}   → {'no se rechaza' if pc['acepta'] else 'se rechaza'} H0")

# Comprobación con SciPy
print("scipy.stats.chisquare:", stats.chisquare(pc["O"]))

sns.barplot(data=tabla_chi, x="intervalo", y="O", color=MORADO)
plt.axhline(pc["E"], color=CORAL, ls="--", lw=2, label="E = 10"); plt.title("Frecuencias observadas"); plt.legend(); plt.show()

### 5.2 Kolmogorov-Smirnov con 10 números
$$D^+ = \max\left(\tfrac{i}{n} - r_{(i)}\right),\quad D^- = \max\left(r_{(i)} - \tfrac{i-1}{n}\right),\quad D = \max(D^+, D^-)$$

In [ ]:
pk = prueba_ks(r10)
tabla_ks = pd.DataFrame({"r(i)": pk["orden"], "i/n": np.arange(1, 11) / 10, "i/n - r(i)": pk["dmas"], "r(i) - (i-1)/n": pk["dmenos"]}, index=pd.RangeIndex(1, 11, name="i"))
display(tabla_ks.T)
print(f"D = {pk['D']:.4f}   crítico D(0.05, 10) = {pk['crit']:.3f}   → {'no se rechaza' if pk['acepta'] else 'se rechaza'} H0")
print("scipy.stats.kstest:", stats.kstest(r10, "uniform"))

plt.step(np.r_[0, pk["orden"], 1], np.r_[0, np.arange(1, 11) / 10, 1], where="post", color=MORADO, lw=2, label="acumulada empírica")
plt.plot([0, 1], [0, 1], color=CORAL, lw=2, label="U(0, 1)")
plt.title("Distribución acumulada: muestra frente a U(0, 1)"); plt.xlabel("r"); plt.legend(); plt.show()

## 6. Pruebas de independencia

### 6.1 Corridas arriba y abajo
$$\mu_C = \frac{2n-1}{3},\qquad \sigma_C^2 = \frac{16n-29}{90},\qquad Z_0 = \frac{C_0 - \mu_C}{\sigma_C}$$

### 6.2 Corridas arriba y abajo de la media
$$\mu_C = \frac{2n_0 n_1}{n} + \frac12,\qquad \sigma_C^2 = \frac{2n_0n_1(2n_0n_1 - n)}{n^2(n-1)}$$

In [ ]:
ca, cm = corridas_arriba_abajo(r20), corridas_media(r20)
print("Secuencia S (arriba y abajo):     ", "".join(map(str, ca["S"])))
print("Secuencia S (respecto a la media):", "".join(map(str, cm["S"])))
pd.DataFrame([
    {"prueba": "Corridas arriba y abajo", "C0": ca["C"], "μ": ca["mu"], "σ²": ca["var"], "Z0": ca["z0"], "no se rechaza H0": ca["acepta"]},
    {"prueba": "Corridas de la media", "C0": cm["C"], "μ": cm["mu"], "σ²": cm["var"], "Z0": cm["z0"], "no se rechaza H0": cm["acepta"]},
]).set_index("prueba")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(range(1, 21), r20, color=TINTA, lw=1, zorder=1)
ax.scatter(range(1, 21), r20, c=[AMBAR] + [VERDE if s else CORAL for s in ca["S"]], s=70, zorder=2)
ax.axhline(0.5, color=MORADO, ls="--", label="0.5")
ax.set(title="Verde: sube · Rojo: baja · línea morada: media teórica", xlabel="i", ylabel="r"); ax.legend(); plt.show()

### 6.3 Prueba de póker con 200 números

Se toman los 5 primeros decimales de cada número y se clasifican en manos. Las categorías con $E_i < 5$ se agrupan.

In [ ]:
print("Primeros números:", [(f"{v:.5f}", mano(v)) for v in r200[:6]])
pp = prueba_poker(r200)
tabla_poker = pd.DataFrame(pp["filas"]).set_index("cat")
display(tabla_poker)
print(f"χ²₀ = {pp['chi0']:.2f}   crítico χ²(0.05, {pp['gl']}) = {pp['crit']:.2f}   → {'no se rechaza' if pp['acepta'] else 'se rechaza'} H0")

tabla_poker[["O", "E"]].plot.bar(color=[MORADO, AMBAR], rot=0)
plt.title("Prueba de póker: observadas frente a esperadas"); plt.show()

---
## 7. Batería completa: 5 generadores, 7 pruebas

In [ ]:
def bateria(r):
    return {"Medias": prueba_medias(r)["acepta"], "Varianza": prueba_varianza(r)["acepta"],
            "Chi-cuadrada": chi2_uniformidad(r)["acepta"], "K-S": prueba_ks(r)["acepta"],
            "Corridas": corridas_arriba_abajo(r)["acepta"], "Corridas media": corridas_media(r)["acepta"],
            "Póker": prueba_poker(r)["acepta"]}

n = 500
generadores = {
    "NumPy (PCG64)": np.random.default_rng(1).random(n),
    "MINSTD": minstd(12345, n),
    "Mixto a=19, c=33, m=100": np.array([f["r"] for f in congruencial(19, 33, 100, 37, n)]),
    "Cuadrados medios 5735": np.array([f["r"] for f in cuadrados_medios(5735, n)]),
    "Aditivo m=100": np.array([f["r"] for f in aditivo([65, 89, 98, 3, 69], 100, n)]),
}
resultados = pd.DataFrame({g: bateria(r) for g, r in generadores.items()}).T
resultados["Total"] = resultados.sum(axis=1)
display(resultados)

plt.figure(figsize=(10, 3.8))
sns.heatmap(resultados.drop(columns="Total").astype(int), annot=resultados.drop(columns="Total").replace({True: "Pasa", False: "No"}),
            fmt="", cmap=sns.color_palette([CORAL, VERDE]), cbar=False, linewidths=1, linecolor="white")
plt.title("Resultado de cada prueba (500 números por generador)"); plt.show()

---
## 8. ¿Qué significa α = 0.05?

Aunque el generador sea bueno, cada prueba **rechaza por azar** cerca del 5 % de las veces. Se aplican las siete pruebas a 300 secuencias de PCG64 y se cuenta la proporción de rechazos.

In [ ]:
rechazos = pd.DataFrame([{k: not v for k, v in bateria(np.random.default_rng(1000 + i).random(300)).items()} for i in range(300)])
tasa = rechazos.mean().rename("proporción de rechazos").reset_index().rename(columns={"index": "prueba"})
sns.barplot(data=tasa, x="prueba", y="proporción de rechazos", color=AZUL)
plt.axhline(0.05, color=CORAL, ls="--", lw=2, label="α = 0.05"); plt.xticks(rotation=20); plt.legend()
plt.title("Rechazos con un generador bueno (300 réplicas)"); plt.show()
print(f"Secuencias que fallan al menos una prueba: {rechazos.any(axis=1).mean():.1%}")

**Observa la prueba de varianza:** casi nunca rechaza. La fórmula con chi-cuadrada supone que los datos son *normales*; con datos uniformes la varianza muestral fluctúa menos de lo que esa fórmula prevé, así que los límites quedan anchos y la prueba es **conservadora**. Sigue siendo útil para detectar desviaciones grandes (por ejemplo, números concentrados cerca de 0.5), pero no es sensible a fallas pequeñas.

Con siete pruebas, una secuencia buena falla al menos una con frecuencia apreciable. Por eso un solo rechazo aislado no condena a un generador: se repite con otras secuencias y se observa si el fallo es sistemático.

---
## 9. Ejercicios

1. **Multiplicador constante.** $a = 4567$, $X_0 = 1234$, $D = 4$: genera cinco $r_i$.
2. **Congruencial aditivo.** Semillas 12, 85, 47, 30, 91 y $m = 100$: genera $X_6$ a $X_{10}$.
3. **Congruencial cuadrático.** $a = 6$, $c = 5$, $m = 16$, $X_0 = 3$: periodo con $b = 7$ y con $b = 9$.
4. **Prueba de medias.** $n = 30$ y $\bar r = 0.5412$, $\alpha = 5\,\%$.
5. **Prueba de varianza.** $n = 30$ y $S^2 = 0.0602$, $\alpha = 5\,\%$.
6. **Chi-cuadrada.** 50 números en 5 intervalos con $O = 8, 12, 9, 13, 8$.
7. **Kolmogorov-Smirnov.** 0.44, 0.81, 0.14, 0.05, 0.93.
8. **Corridas arriba y abajo.** 0.41, 0.68, 0.89, 0.94, 0.74, 0.91, 0.55, 0.62, 0.36, 0.27.
9. **Corridas de la media.** Los mismos 10 números.
10. **Póker.** Clasifica 0.12345, 0.11234, 0.11223, 0.11123, 0.11122, 0.11112, 0.11111, 0.90873, 0.44071, 0.37378.

**Retos con código**

* Encuentra un multiplicador $a$ de 4 dígitos para el método de multiplicador constante que produzca más de 100 valores distintos con $X_0 = 1234$.
* Modifica `prueba_poker` para usar solo 3 decimales (probabilidades: todos diferentes 0.72, un par 0.27, tercia 0.01).
* Implementa la **prueba de series**: divide el cuadrado unitario en 25 celdas y aplica chi-cuadrada a los pares $(r_i, r_{i+1})$.

In [ ]:
# Comprobación de los ejercicios 1 a 3
print("1)", [f["r"] for f in multiplicador_constante(4567, 1234, 5)])
print("2)", [(f["i"], f["Xn"], round(f["r"], 4)) for f in aditivo([12, 85, 47, 30, 91], 100, 5)])
for b in (7, 9):
    print(f"3) b = {b}: periodo =", periodo([f["Xn"] for f in cuadratico(6, b, 5, 16, 3, 50)])[1], "| (b - a) mod 4 =", (b - 6) % 4)

In [ ]:
# Comprobación de los ejercicios 4 a 10
n, z = 30, stats.norm.ppf(0.975)
print(f"4) LI = {0.5 - z/np.sqrt(12*n):.4f}, LS = {0.5 + z/np.sqrt(12*n):.4f}")
li, ls = stats.chi2.ppf(0.025, 29) / 348, stats.chi2.ppf(0.975, 29) / 348
print(f"5) LI = {li:.4f}, LS = {ls:.4f} → 0.0602 {'dentro' if li <= 0.0602 <= ls else 'fuera'}")
O = np.array([8, 12, 9, 13, 8]); print(f"6) χ²₀ = {((O - 10)**2 / 10).sum():.2f}, crítico = {stats.chi2.ppf(0.95, 4):.2f}")
k = prueba_ks([0.44, 0.81, 0.14, 0.05, 0.93]); print(f"7) D+ = {k['Dmas']:.2f}, D- = {k['Dmenos']:.2f}, D = {k['D']:.2f}, crítico = {k['crit']:.3f}")
e8 = [0.41, 0.68, 0.89, 0.94, 0.74, 0.91, 0.55, 0.62, 0.36, 0.27]
a, b = corridas_arriba_abajo(e8), corridas_media(e8)
print(f"8) S = {''.join(map(str, a['S']))}, C0 = {a['C']}, Z0 = {a['z0']:.3f}")
print(f"9) S = {''.join(map(str, b['S']))}, n0 = {b['n0']}, n1 = {b['n1']}, C0 = {b['C']}, Z0 = {b['z0']:.3f}")
print("10)", [(s, mano(float(s))) for s in ["0.12345", "0.11234", "0.11223", "0.11123", "0.11122", "0.11112", "0.11111", "0.90873", "0.44071", "0.37378"]])